P=NP por Holografia CODIGO REAL

In [1]:
"""
stark3sat.py — Mini-librería estilo ZK-STARK para verificar soluciones de 3-SAT.

Componentes reales de un STARK que SÍ están aquí:
  * Compromisos por árbol de Merkle (el prover se compromete a la asignación
    y a la fórmula sin revelarlas por completo).
  * Transformación de Fiat–Shamir (protocolo no interactivo: los índices de
    consulta se derivan del hash de los compromisos, el prover no puede
    elegirlos).
  * Verificación sublineal: el verificador solo abre k cláusulas al azar y
    comprueba caminos de Merkle de longitud log2(n)  →  coste O(k·log n),
    independiente de leer los millones de cláusulas/variables.

Lo que NO está (y en un STARK de producción sí):
  * Aritmetización AIR + test de bajo grado FRI. Sin FRI, este esquema es un
    "spot-check" probabilístico tipo PCP: detecta asignaciones que falsan una
    fracción ε de cláusulas con prob. 1-(1-ε)^k, pero una única cláusula
    falsada puede escapar. Los STARKs reales amplifican ese error con
    codificación Reed–Solomon para lograr solidez plena.
  * Zero-knowledge pleno (habría que enmascarar las aperturas con sal
    aleatoria por hoja; aquí se incluye una sal simple por hoja de asignación).
"""

from __future__ import annotations

import hashlib
import os
import struct
from dataclasses import dataclass, field

HASH = hashlib.sha256
DIGEST = 32


def H(data: bytes) -> bytes:
    return HASH(data).digest()


# ---------------------------------------------------------------------------
# Árbol de Merkle
# ---------------------------------------------------------------------------

class MerkleTree:
    """Árbol de Merkle sobre una lista de hojas (bytes)."""

    def __init__(self, leaves: list[bytes]):
        n = 1
        while n < len(leaves):
            n *= 2
        self.n_leaves = len(leaves)
        self.size = n
        level = [H(b"leaf:" + leaf) for leaf in leaves]
        level += [H(b"pad")] * (n - len(leaves))
        self.levels = [level]
        while len(level) > 1:
            level = [H(level[i] + level[i + 1]) for i in range(0, len(level), 2)]
            self.levels.append(level)

    @property
    def root(self) -> bytes:
        return self.levels[-1][0]

    def open(self, index: int) -> list[bytes]:
        """Camino de autenticación de la hoja `index` (longitud log2(n))."""
        path = []
        for level in self.levels[:-1]:
            path.append(level[index ^ 1])
            index //= 2
        return path

    @staticmethod
    def verify(root: bytes, index: int, leaf: bytes, path: list[bytes]) -> tuple[bool, int]:
        """Devuelve (ok, nº de hashes calculados) — el coste es O(log n)."""
        h = H(b"leaf:" + leaf)
        ops = 1
        for sibling in path:
            h = H(sibling + h) if index & 1 else H(h + sibling)
            index //= 2
            ops += 1
        return h == root, ops


# ---------------------------------------------------------------------------
# Serialización de hojas
# ---------------------------------------------------------------------------

def clause_leaf(clause: tuple[int, int, int]) -> bytes:
    """Cláusula = 3 literales con signo (DIMACS: ±(i+1))."""
    return struct.pack("<3q", *clause)


def assign_leaf(bit: int, salt: bytes) -> bytes:
    """Hoja de asignación con sal (oculta el bit ante quien no la abre)."""
    return bytes([bit]) + salt


# ---------------------------------------------------------------------------
# Fiat–Shamir
# ---------------------------------------------------------------------------

def fiat_shamir_indices(seed: bytes, k: int, domain: int) -> list[int]:
    out, ctr = [], 0
    while len(out) < k:
        d = H(b"fs:" + seed + struct.pack("<Q", ctr))
        ctr += 1
        out.append(int.from_bytes(d[:8], "little") % domain)
    return out


# ---------------------------------------------------------------------------
# Prueba
# ---------------------------------------------------------------------------

@dataclass
class Opening:
    index: int
    leaf: bytes
    path: list[bytes]


@dataclass
class Proof:
    root_formula: bytes
    root_assignment: bytes
    n_vars: int
    n_clauses: int
    k: int
    clause_openings: list[Opening] = field(default_factory=list)
    var_openings: list[list[Opening]] = field(default_factory=list)  # 3 por cláusula

    def size_bytes(self) -> int:
        total = 2 * DIGEST + 24
        for op in self.clause_openings:
            total += 8 + len(op.leaf) + DIGEST * len(op.path)
        for group in self.var_openings:
            for op in group:
                total += 8 + len(op.leaf) + DIGEST * len(op.path)
        return total


# ---------------------------------------------------------------------------
# Prover  (trabajo O(n + m) — el prover SÍ toca todo, como en un STARK real)
# ---------------------------------------------------------------------------

class Prover:
    def __init__(self, clauses: list[tuple[int, int, int]], assignment: list[int]):
        self.clauses = clauses
        self.assignment = assignment
        self.salts = [os.urandom(8) for _ in assignment]
        self.tree_f = MerkleTree([clause_leaf(c) for c in clauses])
        self.tree_a = MerkleTree(
            [assign_leaf(b, s) for b, s in zip(assignment, self.salts)]
        )

    def prove(self, k: int = 96) -> Proof:
        proof = Proof(
            root_formula=self.tree_f.root,
            root_assignment=self.tree_a.root,
            n_vars=len(self.assignment),
            n_clauses=len(self.clauses),
            k=k,
        )
        seed = H(b"seed:" + self.tree_f.root + self.tree_a.root)
        for j in fiat_shamir_indices(seed, k, len(self.clauses)):
            proof.clause_openings.append(
                Opening(j, clause_leaf(self.clauses[j]), self.tree_f.open(j))
            )
            group = []
            for lit in self.clauses[j]:
                v = abs(lit) - 1
                group.append(
                    Opening(
                        v,
                        assign_leaf(self.assignment[v], self.salts[v]),
                        self.tree_a.open(v),
                    )
                )
            proof.var_openings.append(group)
        return proof


# ---------------------------------------------------------------------------
# Verifier  (trabajo O(k · log n) — nunca lee la fórmula ni la asignación)
# ---------------------------------------------------------------------------

class Verifier:
    """Solo conoce: root de la fórmula (digest público del enunciado),
    nº de variables y nº de cláusulas."""

    def __init__(self, root_formula: bytes, n_vars: int, n_clauses: int):
        self.root_formula = root_formula
        self.n_vars = n_vars
        self.n_clauses = n_clauses
        self.hash_ops = 0  # contador de trabajo real

    def verify(self, proof: Proof) -> bool:
        self.hash_ops = 0
        if proof.root_formula != self.root_formula:
            return False
        if proof.n_vars != self.n_vars or proof.n_clauses != self.n_clauses:
            return False

        # Re-derivar los índices de Fiat–Shamir: el prover no pudo elegirlos.
        seed = H(b"seed:" + proof.root_formula + proof.root_assignment)
        expected = fiat_shamir_indices(seed, proof.k, self.n_clauses)
        self.hash_ops += 1 + proof.k

        if len(proof.clause_openings) != proof.k:
            return False

        for j_expected, c_op, group in zip(
            expected, proof.clause_openings, proof.var_openings
        ):
            if c_op.index != j_expected:
                return False
            ok, ops = MerkleTree.verify(
                self.root_formula, c_op.index, c_op.leaf, c_op.path
            )
            self.hash_ops += ops
            if not ok:
                return False
            lits = struct.unpack("<3q", c_op.leaf)

            satisfied = False
            for lit, v_op in zip(lits, group):
                if v_op.index != abs(lit) - 1:
                    return False
                ok, ops = MerkleTree.verify(
                    proof.root_assignment, v_op.index, v_op.leaf, v_op.path
                )
                self.hash_ops += ops
                if not ok:
                    return False
                bit = v_op.leaf[0]
                if (bit == 1 and lit > 0) or (bit == 0 and lit < 0):
                    satisfied = True
            if not satisfied:
                return False  # ¡cláusula falsada al descubierto!
        return True


In [2]:
"""
verify_huge_3sat.py — importa la librería estilo ZK-STARK y verifica una
solución de un 3-SAT ENORME (2 millones de variables, 8 millones de cláusulas)
con trabajo del verificador O(k · log n).
"""

import math
import random
import time



N_VARS = 250_000
N_CLAUSES = 1_000_000
K_QUERIES = 96          # nº de spot-checks (parámetro de seguridad)
random.seed(42)

# ---------------------------------------------------------------------------
# 1. Instancia 3-SAT enorme "plantada": generamos primero la asignación y
#    luego cláusulas garantizadas satisfechas (técnica estándar de benchmark).
# ---------------------------------------------------------------------------
print(f"Generando instancia: {N_VARS:,} variables, {N_CLAUSES:,} cláusulas...")
t0 = time.time()
assignment = [random.getrandbits(1) for _ in range(N_VARS)]

clauses = []
for _ in range(N_CLAUSES):
    vs = random.sample(range(N_VARS), 3)
    lits = []
    for v in vs:
        sign = 1 if random.getrandbits(1) else -1
        lits.append(sign * (v + 1))
    # Forzar que al menos un literal sea verdadero bajo `assignment`
    v0 = vs[0]
    lits[0] = (v0 + 1) if assignment[v0] == 1 else -(v0 + 1)
    clauses.append(tuple(lits))
print(f"  listo en {time.time() - t0:.1f}s")

# ---------------------------------------------------------------------------
# 2. PROVER: se compromete a fórmula y asignación, genera la prueba.
#    (Coste O(n+m) — igual que en un STARK real, el prover paga el trabajo.)
# ---------------------------------------------------------------------------
print("\nProver: construyendo compromisos Merkle y prueba...")
t0 = time.time()
prover = Prover(clauses, assignment)
proof = prover.prove(k=K_QUERIES)
t_prove = time.time() - t0
print(f"  prueba generada en {t_prove:.1f}s")
print(f"  tamaño de la prueba: {proof.size_bytes() / 1024:.1f} KiB "
      f"(vs {N_CLAUSES * 24 / 1e6:.0f} MB de la fórmula cruda)")

# El digest público del enunciado (raíz Merkle de la fórmula) es lo ÚNICO
# que el verificador necesita conocer de la instancia:
root_formula = prover.tree_f.root

# ---------------------------------------------------------------------------
# 3. VERIFIER: solo usa la raíz pública + la prueba. Nunca lee los 8M de
#    cláusulas ni los 2M de bits. Trabajo: O(k · log n).
# ---------------------------------------------------------------------------
print("\nVerifier: comprobando la prueba...")
t0 = time.time()
verifier = Verifier(root_formula, N_VARS, N_CLAUSES)
ok = verifier.verify(proof)
t_verify = time.time() - t0

log_m = math.ceil(math.log2(N_CLAUSES))
log_n = math.ceil(math.log2(N_VARS))
bound = K_QUERIES * (1 + log_m + 3 * (1 + log_n)) + K_QUERIES + 1

print(f"  resultado: {'ACEPTADA ✓' if ok else 'RECHAZADA ✗'}")
print(f"  tiempo verificación: {t_verify * 1000:.1f} ms  "
      f"(prover tardó {t_prove:.1f} s → {t_prove / t_verify:,.0f}× más)")
print(f"  hashes calculados por el verificador: {verifier.hash_ops:,} "
      f"(cota teórica k·log: ~{bound:,})")
print(f"  comparación: leer la instancia entera serían {N_CLAUSES + N_VARS:,} "
      f"elementos; el verificador tocó {verifier.hash_ops:,} hashes.")

# ---------------------------------------------------------------------------
# 4. Test de solidez: un prover tramposo con una asignación mayormente mala.
# ---------------------------------------------------------------------------
print("\nTest de solidez: prover tramposo (asignación aleatoria)...")
bad_assignment = [random.getrandbits(1) for _ in range(N_VARS)]
cheater = Prover(clauses, bad_assignment)
bad_proof = cheater.prove(k=K_QUERIES)
rejected = not verifier.verify(bad_proof)
print(f"  prueba tramposa {'RECHAZADA ✓' if rejected else 'ACEPTADA ✗ (fallo)'}")
# Con ~1/8 de cláusulas falsadas y k=96: prob. de escape ≈ (7/8)^96 ≈ 2.7e-6.

# ---------------------------------------------------------------------------
# 5. Test de manipulación: alterar una apertura de la prueba válida.
# ---------------------------------------------------------------------------
print("\nTest de integridad: manipulando una hoja de la prueba válida...")
tampered_leaf = clause_leaf((1, 2, 3))
original = proof.clause_openings[0].leaf
proof.clause_openings[0].leaf = tampered_leaf
tam_rejected = not verifier.verify(proof)
proof.clause_openings[0].leaf = original
print(f"  prueba manipulada {'RECHAZADA ✓' if tam_rejected else 'ACEPTADA ✗ (fallo)'}")

assert ok and rejected and tam_rejected
print("\nTodos los tests pasaron.")


Generando instancia: 250,000 variables, 1,000,000 cláusulas...
  listo en 2.5s

Prover: construyendo compromisos Merkle y prueba...
  prueba generada en 1.6s
  tamaño de la prueba: 229.9 KiB (vs 24 MB de la fórmula cruda)

Verifier: comprobando la prueba...
  resultado: ACEPTADA ✓
  tiempo verificación: 4.6 ms  (prover tardó 1.6 s → 356× más)
  hashes calculados por el verificador: 7,585 (cota teórica k·log: ~7,585)
  comparación: leer la instancia entera serían 1,250,000 elementos; el verificador tocó 7,585 hashes.

Test de solidez: prover tramposo (asignación aleatoria)...
  prueba tramposa RECHAZADA ✓

Test de integridad: manipulando una hoja de la prueba válida...
  prueba manipulada RECHAZADA ✓

Todos los tests pasaron.


In [3]:
import random
import os
import contextlib

# Se asume que Prover, Verifier y K_QUERIES ya están definidos


def simplify_cnf(cnf, assignment):
    """
    Simplifica una CNF con una asignación parcial.
    """

    new_cnf = []

    for clause in cnf:
        remaining = []
        satisfied = False

        for lit in clause:
            var = abs(lit)

            if var not in assignment:
                remaining.append(lit)
                continue

            value = assignment[var]

            if (lit > 0 and value) or (lit < 0 and not value):
                satisfied = True
                break

        if satisfied:
            continue

        new_cnf.append(remaining)

    return new_cnf



def verify_assignment_with_zkstark(cnf, assignment, nvars, k_queries=K_QUERIES):

    assignment_list = [
        1 if assignment[i] else 0
        for i in range(1, nvars + 1)
    ]

    with open(os.devnull, "w") as fnull:
        with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):

            prover = Prover(cnf, assignment_list)

            proof = prover.prove(k=k_queries)

            verifier = Verifier(
                root_formula=prover.tree_f.root,
                n_vars=nvars,
                n_clauses=len(cnf)
            )

            result = verifier.verify(proof)

    return result



def autoreduce_first_variable(cnf, nvars, k_queries=K_QUERIES):

    assignment = {}

    for v in range(2, nvars + 1):
        assignment[v] = random.choice([True, False])

    assignment[1] = True


    accepted = verify_assignment_with_zkstark(
        cnf=cnf,
        assignment=assignment,
        nvars=nvars,
        k_queries=k_queries
    )

    return accepted



def autoreduce_all_variables(cnf, nvars, k_queries=K_QUERIES):

    final_assignment = {}

    for forced_var in range(1, nvars + 1):

        candidate_assignment = {}

        for v in range(1, nvars + 1):
            candidate_assignment[v] = random.choice([True, False])

        candidate_assignment[forced_var] = True


        accepted = verify_assignment_with_zkstark(
            cnf=cnf,
            assignment=candidate_assignment,
            nvars=nvars,
            k_queries=k_queries
        )


        if accepted:
            final_assignment[forced_var] = True
        else:
            final_assignment[forced_var] = False


    return final_assignment



cnf = [
    [1, 2, 2],
    [-1, 3, 3],
    [-2, -3, -3]
]


N_VARS = 3


resultado_x1 = autoreduce_first_variable(
    cnf,
    N_VARS,
    k_queries=K_QUERIES
)


asignacion_final = autoreduce_all_variables(
    cnf,
    N_VARS,
    k_queries=K_QUERIES
)

In [ ]:
import sys
import os
import struct
from google.colab import files

# ==========================================
# 1. FUNCIÓN CLAUSE_LEAF (CON FIX DE STRUCT)
# ==========================================
import struct

def clause_leaf(clause) -> bytes:
    """
    Empaqueta los literales de la cláusula forzando EXACTAMENTE 3 literales (24 bytes)
    para que sea compatible con el unpack("<3q") del verificador.
    """
    if len(clause) == 1 and isinstance(clause[0], (list, tuple)):
        clause = clause[0]

    clause_flat = [int(x) for x in clause]

    # Rellenar si tiene menos de 3 literales (ej: [1, 2] -> [1, 2, 2])
    while len(clause_flat) < 3:
        clause_flat.append(clause_flat[-1])

    # Truncar si tiene más de 3 (para evitar que truene el buffer de 24 bytes)
    if len(clause_flat) > 3:
        clause_flat = clause_flat[:3]

    return struct.pack("<3q", *clause_flat)


# ==========================================
# 2. PARSER DIMACS CNF
# ==========================================
def cargar_dimacs_cnf_desde_texto(contenido_texto):
    """
    Lee una fórmula en formato DIMACS CNF desde una cadena de texto.
    Retorna la lista de cláusulas (cnf) y el número de variables (n_vars).
    """
    cnf = []
    n_vars = 0

    for linea in contenido_texto.splitlines():
        linea = linea.strip()
        # Omitir comentarios o líneas vacías
        if not linea or linea.startswith('c'):
            continue

        tokens = linea.split()

        # Encabezado: p cnf NUM_VARS NUM_CLAUSULAS
        if tokens[0] == 'p':
            n_vars = int(tokens[2])
            continue

        # Parsear literales de la cláusula
        literales = [int(t) for t in tokens if t != '0']
        if literales:
            # Convertimos a tupla limpia
            cnf.append(tuple(literales))

    return cnf, n_vars


# ==========================================
# 3. EXPORTADOR A FORMATO ESTÁNDAR
# ==========================================
def generar_texto_solucion(asignacion):
    """
    Genera el string con la salida en formato DIMACS estándar.
    """
    if asignacion is None:
        return "UNSAT\n"

    lineas = ["SAT"]
    literales_salida = []

    if isinstance(asignacion, dict):
        for var, val in asignacion.items():
            literales_salida.append(str(var) if val else str(-var))
    elif isinstance(asignacion, (list, tuple)):
        literales_salida = [str(x) for x in asignacion]

    lineas.append("v " + " ".join(literales_salida) + " 0")
    return "\n".join(lineas) + "\n"


# ==========================================
# 4. FLUJO PRINCIPAL EN COLAB
# ==========================================

archivos_subidos = files.upload()

for nombre_archivo, contenido in archivos_subidos.items():

    if nombre_archivo.endswith('.cnf'):

        texto_cnf = contenido.decode('utf-8')
        cnf, N_VARS = cargar_dimacs_cnf_desde_texto(texto_cnf)


        # Progreso silencioso
        print("Procesando fórmula CNF...")


        # Ejecutar reducción con progreso discreto
        asignacion_final = {}

        for i in range(1, N_VARS + 1):

            candidato = {}

            for v in range(1, N_VARS + 1):
                candidato[v] = random.choice([True, False])

            candidato[i] = True


            aceptado = verify_assignment_with_zkstark(
                cnf=cnf,
                assignment=candidato,
                nvars=N_VARS,
                k_queries=K_QUERIES
            )


            asignacion_final[i] = aceptado


            # Actualiza una sola línea
            porcentaje = (i / N_VARS) * 100
            print(
                f"\rProgreso: {i}/{N_VARS} variables ({porcentaje:.1f}%)",
                end="",
                flush=True
            )


        print("\nProceso terminado.")


        nombre_salida = "solution.txt"

        resultado_txt = generar_texto_solucion(asignacion_final)


        with open(nombre_salida, "w") as f:
            f.write(resultado_txt)


        print("Archivo generado:", nombre_salida)

        files.download(nombre_salida)

Saving sha3_collision (3).cnf to sha3_collision (3).cnf
Procesando fórmula CNF...
Progreso: 235/310636 variables (0.1%)